# Advanced Retrieval Augmented Generation with LangChain

## Initial Setup

In [1]:
# !uv pip install sentence-transformers

In [2]:
import os, json, re, getpass, warnings
import numpy as np
from dotenv import load_dotenv
from uuid import uuid4
from IPython.display import display, Markdown

In [3]:
warnings.filterwarnings('ignore')
load_dotenv(override=True)

True

In [4]:
#Check for Groq API Key
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ API Key: ")

In [5]:
#Set Hugging Face token (as we will be using some of Hugging Face's functionalities
from huggingface_hub.hf_api import HfFolder
HfFolder.save_token(os.environ["HF_TOKEN"])

## Defining Components

In [6]:
# Question
question = "What is the PGP AI & DS at Jio Institute all about?"

### Chat Model

In [24]:
from langchain.chat_models import init_chat_model

model_name = "openai/gpt-oss-120b"
llm = init_chat_model(model_name, model_provider="groq")

### Embedding Model

In [8]:
!ollama pull qwen3-embedding:0.6b

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling 06507c7b4268: 100% ▕██████████████████▏ 639 MB                         
pulling 9202febed9e2: 100% ▕██████████████████▏  266 B                         
verifying sha256 digest 
writing manifest 
success 


In [9]:
from langchain_ollama import OllamaEmbeddings

embeddings_model = OllamaEmbeddings(model="nomic-embed-text")

### Vector Store

In [10]:
#Import library
from langchain_chroma import Chroma

In [11]:
#Create a vector store
vector_store_chroma = Chroma(
    collection_name="advanced-rag",
    embedding_function=embeddings_model,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

## Indexing

### Loading Documents

In [12]:
import bs4 #import Beautiful Soup
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [13]:
# Only keep the main content from the full HTML
bs4_strainer = bs4.SoupStrainer(class_=("node__content clearfix", "col-md-9 pl-lg-5"))
loader = WebBaseLoader(
    web_paths=("https://www.jioinstitute.edu.in/about/",
    "https://www.jioinstitute.edu.in/academics/artificial-intelligence-data-science"),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 2

print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 4361


In [14]:
# --- Post-processing to clean up excessive newlines and whitespace ---
for i, doc in enumerate(docs):
    if doc.page_content:
        # 1. Replace multiple newlines with a single newline
        cleaned_content = re.sub(r'\n\s*\n', '\n\n', doc.page_content)
        # 2. Replace multiple spaces with a single space
        cleaned_content = re.sub(r' {2,}', ' ', cleaned_content)
        # 3. Strip leading/trailing whitespace from each line
        cleaned_content = '\n'.join([line.strip() for line in cleaned_content.split('\n')])
        # 4. Remove leading/trailing whitespace from the whole string
        cleaned_content = cleaned_content.strip()
    
        docs[i].page_content = cleaned_content
# --- End of post-processing ---

In [15]:
print(f"Total characters: {len(docs[0].page_content)}")
print("\n--- Cleaned Content Snippet ---")
print(docs[0].page_content[:1000]) # Print a snippet to verify

Total characters: 3702

--- Cleaned Content Snippet ---
About Us

Jio Institute is a multidisciplinary higher education institute set up as a philanthropic initiative by the Reliance Group. The Institute is dedicated to the pursuit of excellence by bringing together global scholars and thought leaders and providing an enriching student experience through world-class education, and a culture of research and innovation.

Our Story
Pursuit of excellence in academics, research and innovation.
We stand at the confluence of the best higher education practices from India and the world. The institute aims to nurture students’ aspirations, and provide a platform to their entrepreneurial spirit.

Read more

Our Vision
In sync with global aspirations. In step with changing times.
We envisage to be a world-class higher education institute through our multi-disciplinary academic programmes, robust research endeavours and a culture of innovation and entrepreneurship.

Read more

Growth Plan
Well tho

### Splitting Documents

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 63 sub-documents.


In [17]:
for i in range(len(all_splits)):
    print(f"\n--- Split{i} ---\n")
    print(all_splits[i].page_content)


--- Split0 ---

About Us

Jio Institute is a multidisciplinary higher education institute set up as a philanthropic initiative by the Reliance Group. The Institute is dedicated to the pursuit of excellence by bringing together global scholars and thought leaders and providing an enriching student experience through world-class education, and a culture of research and innovation.

Our Story
Pursuit of excellence in academics, research and innovation.
We stand at the confluence of the best higher education practices from India and the world. The institute aims to nurture students’ aspirations, and provide a platform to their entrepreneurial spirit.

Read more

Our Vision
In sync with global aspirations. In step with changing times.
We envisage to be a world-class higher education institute through our multi-disciplinary academic programmes, robust research endeavours and a culture of innovation and entrepreneurship.

Read more

--- Split1 ---

Read more

Growth Plan
Well thought out gro

### Storing in Vector Store

In [18]:
#Add first 10 chunks to vector DB
uuids = [str(uuid4()) for _ in range(len(all_splits))] #Universally unique identifier
vector_store_chroma.add_documents(documents=all_splits[:10], ids=uuids[:10])

['01d92160-7a35-4f64-850e-028c05c17024',
 'f3bee962-c923-4d32-a8b6-001e1852cc77',
 'b15a411c-68c4-42e8-88ff-905f580239be',
 '145d9607-f262-4230-8d26-d2c787984e37',
 '82ebbf01-8a4f-47cf-a2be-4d9f9219923d',
 '86885783-5e92-4476-8e6f-4e9109727e7d',
 '8430ae77-ab91-47fd-9639-433e4f221ccb',
 '1331cb16-860f-425a-babe-b9e22500f5a9',
 '06debd1d-cde1-41e2-bb8e-6a0119974b1e',
 '2b3d4dbb-aed9-4d6d-b27d-be3fd5b114db']

## Advanced Retrieval and Reranking Strategies

### Multi Query Retrieval

Retrieval may produce different results with subtle changes in query wording, or if the embeddings do not capture the semantics of the data well. Prompt engineering / tuning is sometimes done to manually address these problems, but can be tedious.

The [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) automates the process of prompt tuning by using an LLM to generate multiple queries from different perspectives for a given user input query. For each query, it retrieves a set of relevant documents and takes the unique union across all queries to get a larger set of potentially relevant documents.

In [26]:
#Import libraries
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging # Set logging for the queries

In [27]:
#Initialize retriever
retriever = vector_store_chroma.as_retriever(search_type="similarity",
                                                search_kwargs={"k": 2})

In [28]:
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever, 
    llm=llm,
    include_original=True
)

In [29]:
logging.basicConfig()
# so we can see what queries are generated by the LLM
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

In [30]:
docs = mq_retriever.invoke(question)
docs

INFO:langchain.retrievers.multi_query:Generated queries: ['What does the PGP AI & DS program at Jio Institute cover?  ', "Can you explain the curriculum and focus areas of Jio Institute's PGP AI & DS?  ", 'What are the details and objectives of the PGP AI & DS offered by Jio Institute?']


[Document(id='86885783-5e92-4476-8e6f-4e9109727e7d', metadata={'start_index': 0, 'source': 'https://www.jioinstitute.edu.in/academics/artificial-intelligence-data-science'}, page_content='PGP in Artificial Intelligence & Data Science\n\nThis postgraduate programme has a comprehensive and rigorous curriculum that covers foundation and advanced courses to train future-ready full-stack data scientists and AI architects.\xa0\xa0Apply Now \xa0Download Brochure'),
 Document(id='06debd1d-cde1-41e2-bb8e-6a0119974b1e', metadata={'source': 'https://www.jioinstitute.edu.in/academics/artificial-intelligence-data-science', 'start_index': 1492}, page_content='PGP in Artificial Intelligence & Data Science\n\nLeadership\nFaculty\nAdvisors\nCurriculum\nTools\nHighlights\nAdmissions\nFAQ\nBrochure\n\nPGP in Artificial Intelligence & Data Science\n\nChoose your purpose\n\nLeadership\nFaculty\nAdvisors\nCurriculum\nTools\nHighlights\nAdmissions\nFAQ\nBrochure\n\nProgramme Leadership\n\nDr. Larry Birnbaum\

### Chained Retrieval with Reranker

This strategy uses a chain of multiple retrievers sequentially to get to the most relevant documents. The following is the flow:

*Similarity Retrieval → Reranker Model Retrieval*

**What are rerankers?**

- Rerankers are fine-tuned cross-encoder transformer models
- These models take in a pair of documents (Query, Document) and return back a relevance score
- Models fine-tuned on more pairs and released recently will usually be better

In [31]:
#Import libraries
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever

In [32]:
# Retriever 1 - simple cosine distance based retriever
retriever = vector_store_chroma.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 3})

In [33]:
# Download an open-source reranker model - cross-encoder/qnli-electra-base
reranker = HuggingFaceCrossEncoder(model_name="cross-encoder/qnli-electra-base")
reranker_compressor = CrossEncoderReranker(model=reranker, top_n=2)

In [34]:
# Retriever 2 - Uses a Reranker model to rerank retrieval results from the previous retriever
final_retriever = ContextualCompressionRetriever(
    base_compressor=reranker_compressor,
    base_retriever=retriever
)

In [35]:
docs = final_retriever.invoke(question)
docs

[Document(id='86885783-5e92-4476-8e6f-4e9109727e7d', metadata={'source': 'https://www.jioinstitute.edu.in/academics/artificial-intelligence-data-science', 'start_index': 0}, page_content='PGP in Artificial Intelligence & Data Science\n\nThis postgraduate programme has a comprehensive and rigorous curriculum that covers foundation and advanced courses to train future-ready full-stack data scientists and AI architects.\xa0\xa0Apply Now \xa0Download Brochure'),
 Document(id='01d92160-7a35-4f64-850e-028c05c17024', metadata={'start_index': 0, 'source': 'https://www.jioinstitute.edu.in/about/'}, page_content='About Us\n\nJio Institute is a multidisciplinary higher education institute set up as a philanthropic initiative by the Reliance Group. The Institute is dedicated to the pursuit of excellence by bringing together global scholars and thought leaders and providing an enriching student experience through world-class education, and a culture of research and innovation.\n\nOur Story\nPursuit

## Context Compression Strategies

Here, we'll explore **LLM prompt-based context compression** strategies. The context compression can happen in the form of:

- **Extractor**: Remove parts of the content of retrieved documents which are not relevant to the query. This is done by extracting only relevant parts of the document to the given query

- **Filter**: Filter out documents which are not relevant to the given query but do not remove content from the document

Good to also read about [Microsoft LLMLingua Prompt
Compression](https://www.microsoft.com/en-us/research/blog/llmlingua-innovating-llm-efficiency-with-prompt-compression/).

### LLMChainExtractor

Here we look at `LLMChainExtractor`, which will iterate over the initially returned documents and extract from each only the content that is relevant to the query. Totally irrelevant documents might also be dropped.

In [36]:
#Import libraries
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [37]:
#Initialize retriever
retriever = vector_store_chroma.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 3})

In [38]:
# Extracts from each document only the content that is relevant to the query
compressor = LLMChainExtractor.from_llm(llm=llm)

In [39]:
# Retrieves the documents similar to query and then applies the compressor
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=retriever
)

In [40]:
docs = compression_retriever.invoke(question)
docs

[Document(metadata={'start_index': 0, 'source': 'https://www.jioinstitute.edu.in/academics/artificial-intelligence-data-science'}, page_content='PGP in Artificial Intelligence & Data Science\n\nThis postgraduate programme has a comprehensive and rigorous curriculum that covers foundation and advanced courses to train future-ready full-stack data scientists and AI architects.\xa0\xa0Apply Now \xa0Download Brochure'),
 Document(metadata={'source': 'https://www.jioinstitute.edu.in/academics/artificial-intelligence-data-science', 'start_index': 1492}, page_content='PGP in Artificial Intelligence & Data Science\n\nLeadership\nFaculty\nAdvisors\nCurriculum\nTools\nHighlights\nAdmissions\nFAQ\nBrochure\n\nPGP in Artificial Intelligence & Data Science\n\nChoose your purpose\n\nLeadership\nFaculty\nAdvisors\nCurriculum\nTools\nHighlights\nAdmissions\nFAQ\nBrochure\n\nProgramme Leadership\n\nDr. Larry Birnbaum\n\nProfessor, Computer Science, Northwestern University, USA\n\nSee Profile\n\nDr. Sha

### LLMChainFilter

The `LLMChainFilter` is slightly simpler but more robust compressor that uses an LLM chain to decide which of the initially retrieved documents to filter out and which ones to return, without manipulating the document contents.

In [41]:
#Import library
from langchain.retrievers.document_compressors import LLMChainFilter

In [42]:
#Initialize retriever
retriever = vector_store_chroma.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 3})

In [43]:
# Decides which of the initially retrieved documents to filter out and which ones to return
_filter = LLMChainFilter.from_llm(llm=llm)

In [44]:
# Retrieves the documents similar to query and then applies the filter
compression_retriever = ContextualCompressionRetriever(
    base_compressor=_filter, base_retriever=retriever
)

In [45]:
docs = compression_retriever.invoke(question)
docs

[Document(id='86885783-5e92-4476-8e6f-4e9109727e7d', metadata={'source': 'https://www.jioinstitute.edu.in/academics/artificial-intelligence-data-science', 'start_index': 0}, page_content='PGP in Artificial Intelligence & Data Science\n\nThis postgraduate programme has a comprehensive and rigorous curriculum that covers foundation and advanced courses to train future-ready full-stack data scientists and AI architects.\xa0\xa0Apply Now \xa0Download Brochure'),
 Document(id='06debd1d-cde1-41e2-bb8e-6a0119974b1e', metadata={'start_index': 1492, 'source': 'https://www.jioinstitute.edu.in/academics/artificial-intelligence-data-science'}, page_content='PGP in Artificial Intelligence & Data Science\n\nLeadership\nFaculty\nAdvisors\nCurriculum\nTools\nHighlights\nAdmissions\nFAQ\nBrochure\n\nPGP in Artificial Intelligence & Data Science\n\nChoose your purpose\n\nLeadership\nFaculty\nAdvisors\nCurriculum\nTools\nHighlights\nAdmissions\nFAQ\nBrochure\n\nProgramme Leadership\n\nDr. Larry Birnbaum\